# ESMValTool Integration — ACCESS-MOPPy

This notebook demonstrates how to use the `access_moppy.esmval` subpackage to
**CMORise ACCESS-ESM1.6 output on-the-fly for an ESMValTool recipe**.

It shows the Python API step by step so you can understand what each component
does before running the full `moppy-esmval-run` CLI command.

### What you need
- An ESMValTool YAML recipe that references `dataset: ACCESS-ESM1-6`
  and `project: CMIP6`
- A directory of raw ACCESS-ESM1.6 output (UM atmosphere files,
  MOM5 ocean files, CICE sea-ice files)
- A writable cache directory where CMORised CMIP-DRS files will be stored

## 1. Imports and configuration

Install with:
```bash
pip install 'access_moppy[esmval]'
```

In [ ]:
from pathlib import Path

from access_moppy.esmval import (
    CMORiseOrchestrator,
    RecipeReader,
    VariableIndex,
)
from access_moppy.esmval.config_gen import write_esmval_config

# ── Edit these paths for your environment ──────────────────────────────────
RECIPE = Path("my_recipe.yml")  # path to an ESMValTool recipe
INPUT_ROOT = Path("/g/data/p73/archive/CMIP7/ACCESS-ESM1-6/historical/MyRun")
CACHE_DIR = Path.home() / ".cache" / "moppy-esmval"
# ───────────────────────────────────────────────────────────────────────────

CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Cache directory: {CACHE_DIR}")

## 2. Write a sample recipe

If you don't have a recipe handy, the cell below writes a minimal one.
Normally you would already have an ESMValTool recipe — skip this cell if so.

In [ ]:
sample_recipe = """\
documentation:
  title: Sample recipe for ACCESS-MOPPy ESMValTool integration demo
  description: Minimal recipe that needs ACCESS-ESM1-6 surface temperature.
  authors:
    - anonymous

datasets:
  - {dataset: ACCESS-ESM1-6, project: CMIP6, exp: historical,
     ensemble: r1i1p1f1, grid: gn, timerange: '2000/2005'}

diagnostics:
  surface_temperature:
    variables:
      tas:
        mip: Amon
    scripts:
      null_script:
        script: null
"""

RECIPE.write_text(sample_recipe)
print(f"Recipe written to: {RECIPE.resolve()}")

## 3. Parse the recipe with `RecipeReader`

`RecipeReader` scans all `datasets` and `diagnostic.datasets` blocks, keeps
entries with `project: CMIP6` and `dataset: ACCESS-ESM1-6`, and returns a
deduplicated list of `CMORTask` objects — one per `(mip, short_name, experiment,
ensemble, timerange)` combination.

In [ ]:
reader = RecipeReader(RECIPE)
tasks = reader.tasks

print(f"Found {len(tasks)} CMORisation task(s):\n")
for t in tasks:
    print(
        f"  {t.mip}.{t.short_name}  exp={t.experiment_id}"
        f"  variant={t.variant_label}  time={t.timerange}"
    )

## 4. Check supported variables with `VariableIndex`

`VariableIndex` reads the ACCESS-ESM1.6 mapping JSON bundled with
ACCESS-MOPPy and indexes all supported `(mip, short_name)` pairs.

In [ ]:
idx = VariableIndex()  # uses default ACCESS-ESM1.6 mappings

# Check each task
for t in tasks:
    supported = idx.is_supported(t.mip, t.short_name)
    entry = idx.get(t.mip, t.short_name)
    print(f"  {t.mip}.{t.short_name:20s}  supported={supported}", end="")
    if entry:
        print(
            f"  compound_name={entry.compound_name!r}" f"  component={entry.component}"
        )
    else:
        print("  (no mapping found)")

# List variables not (yet) in the mapping:
unsupported = idx.unsupported([(t.mip, t.short_name) for t in tasks])
if unsupported:
    print("\nUnsupported variables:")
    for mip, sn in unsupported:
        print(f"  {mip}.{sn}")
else:
    print("\nAll requested variables are supported.")

## 5. Run CMORisation with `CMORiseOrchestrator`

The orchestrator combines all the steps:
1. Call `RecipeReader` to get tasks
2. Use `VariableIndex` to map each task to its compound name
3. Use `RawFileFinder` to locate raw ACCESS files on disk
4. Run `ACCESS_ESM_CMORiser` (skipping already-cached output)
5. Return a `TaskResult` list you can inspect

> **Note:** Set `dry_run=True` first to see what would happen without
> actually running CMORisation.

In [ ]:
orch = CMORiseOrchestrator(
    input_root=INPUT_ROOT,
    cache_dir=CACHE_DIR,
    workers=1,
    dry_run=True,  # <-- change to False to actually run
)

results = orch.prepare_recipe(RECIPE)
CMORiseOrchestrator.summarise(results)

In [ ]:
# Uncomment to actually CMORise:

# orch_real = CMORiseOrchestrator(
#     input_root=INPUT_ROOT,
#     cache_dir=CACHE_DIR,
#     workers=4,     # parallelism
#     dry_run=False,
# )
# results = orch_real.prepare_recipe(RECIPE)
# CMORiseOrchestrator.summarise(results)

## 6. Write the ESMValCore config overlay

This generates a small YAML file that points ESMValCore at the
CMORised cache.  The config file is written to `~/.config/esmvaltool/moppy-esmval-data.yml`, which ESMValCore 2.14+ picks up automatically.

In [ ]:
config_path = write_esmval_config(CACHE_DIR)
print(f"Config overlay written to: {config_path}")
print()
print(config_path.read_text())

## 7. Run ESMValTool

Open a terminal and run:

```bash
esmvaltool run my_recipe.yml
```

The ESMValCore config written by the previous step is picked up
automatically from `~/.config/esmvaltool/` — no `--config` flag needed.

Or from the notebook:

In [ ]:
# Uncomment to run ESMValTool directly from the notebook:
# result = subprocess.run(
#     ["esmvaltool", "run", str(RECIPE)],
#     check=False,
# )
# print("Return code:", result.returncode)

## 8. One-step CLI equivalent

Everything above is equivalent to the single CLI command:

```bash
moppy-esmval-run my_recipe.yml \
    --input-root /g/data/p73/archive/.../MyRun \
    --cache-dir ~/.cache/moppy-esmval \
    --workers 4
```

Or if ESMValTool is installed alongside ACCESS-MOPPy:

```bash
esmvaltool cmorise my_recipe.yml \
    --input_root /g/data/p73/archive/.../MyRun \
    --cache_dir ~/.cache/moppy-esmval

esmvaltool run my_recipe.yml
```

### Custom file-pattern overrides

If your archive uses a non-standard directory layout, pass pattern
overrides:

```bash
moppy-esmval-prepare my_recipe.yml \
    --input-root /data/archive/MyRun \
    --cache-dir ~/.cache/moppy-esmval \
    --pattern "Amon.tas:output[0-4]*/atmosphere/netCDF/*mon.nc" \
    --pattern "Omon.tos:output[0-4]*/ocean/ocean-2d-surface_temp*.nc"
```

See the
[ESMValTool Integration docs](../docs/source/esmvaltool_integration.rst)
for a full reference.